# 00 — What Is Bearing?

**Bearing** is the direction from one point to another, measured as an angle clockwise from North.

That sentence contains two things that will immediately trip you up: *clockwise* and *from North*. Most of the bearing-related bugs students write come from forgetting one of those two words. This notebook builds the intuition before any formula appears.

In [ ]:
import json
from pathlib import Path

with open(Path("data/bearing_examples.json")) as f:
    examples = json.load(f)

print("Data loaded:")
print(f"  Cardinal directions:      {len(examples['cardinal_directions'])}")
print(f"  Intercardinal directions: {len(examples['intercardinal_directions'])}")
print(f"  City pairs:               {len(examples['city_pairs'])}")
print(f"  Common mistakes:          {len(examples['common_mistakes'])}")

## 1. What Is Bearing?

Bearing answers the question: **if I am standing at point A and facing point B, which direction am I facing?**

The answer is always a single number between 0° and 360°, measured clockwise starting from North.

```
        N (0°)
        |
W ------+------ E
(270°)  |       (90°)
        |
        S (180°)
```

A few things to nail down immediately:

- Bearing is **always measured from North**, not from the positive x-axis
- Bearing increases **clockwise** — going from North toward East
- The result is always in the range **[0°, 360°)** — 360° wraps back to 0°
- Bearing describes a **direction**, not a turn — it is an absolute heading, not a delta

## 2. Compass Convention vs. Math Convention

This is where most bearing bugs are born.

In a standard math class, angles are measured starting from the **positive x-axis (East)** and increase **counter-clockwise**. This is how `math.atan2` works in Python. It is how unit circles are taught. It is wrong for navigation.

Compass bearings do the opposite on both counts.

| Property      | Math convention         | Compass convention      |
|---------------|-------------------------|-------------------------|
| Zero angle    | East (positive x-axis)  | North                   |
| Direction     | Counter-clockwise        | Clockwise               |
| Range         | (-180°, 180°]           | [0°, 360°)              |
| Example: East | 0°                      | 90°                     |
| Example: West | 180° (or -180°)         | 270°                    |

When you compute a bearing from coordinates, you will use `math.atan2` — which returns a math angle. Converting it to a compass bearing requires two steps: convert to degrees, then normalize and rotate. The next notebook covers that formula. For now, just know the mismatch exists and is the source of the "my bearing is negative" or "my East and West are swapped" class of bugs.

In [ ]:
# Print the common mistakes from the data file
for m in examples["common_mistakes"]:
    print(f"Mistake: {m['mistake']}")
    if "math_convention" in m:
        print(f"  Math:    {m['math_convention']}")
        print(f"  Bearing: {m['bearing_convention']}")
        print(f"  Result:  {m['consequence']}")
    if "example" in m:
        print(f"  Example: {m['example']}")
        print(f"  Fix:     {m['fix']}")
    print()

## 3. Cardinal and Intercardinal Directions

The eight standard compass points and their bearings:

In [ ]:
# Merge cardinal and intercardinal, sort by bearing, print as a table
all_directions = examples["cardinal_directions"] + examples["intercardinal_directions"]
all_directions.sort(key=lambda d: d["bearing"])

print(f"{'Bearing':>8}  {'Name':<12}  {'Description'}")
print("-" * 48)
for d in all_directions:
    desc = d.get("description", "")
    print(f"{d['bearing']:>7}°  {d['name']:<12}  {desc}")

A bearing of `270°` is West — not `-90°`. Both describe the same direction, but only `270°` is a valid compass bearing. This is why the normalization step `(degrees + 360) % 360` matters: it converts any negative angle into its positive equivalent.

Some useful sanity checks to keep in your head:

- Bearing less than 90°: target is generally **north** of you (with some east)
- Bearing between 90° and 180°: target is **east and south**
- Bearing between 180° and 270°: target is **south and west**
- Bearing greater than 270°: target is **west and north**

## 4. Visualizing Bearings

A bearing of `45°` means "halfway between North and East." A bearing of `315°` means "halfway between North and West (coming back around clockwise from 270°)."

The plot below places the eight compass directions around a circle to make the clockwise-from-North convention visual. Run it and use it as a reference for the exercises.

In [ ]:
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import math

fig, ax = plt.subplots(figsize=(5, 5))
ax.set_aspect("equal")
ax.set_xlim(-1.5, 1.5)
ax.set_ylim(-1.5, 1.5)
ax.axis("off")
ax.set_title("Compass Rose — Bearing (clockwise from North)", fontsize=11, pad=12)

# Outer circle
circle = plt.Circle((0, 0), 1.0, fill=False, color="#555555", linewidth=1.5)
ax.add_patch(circle)

# All eight compass points — (label, bearing_degrees)
directions = (
    examples["cardinal_directions"] + examples["intercardinal_directions"]
)

for d in directions:
    bearing  = d["bearing"]
    label    = d["name"]
    angle_math = math.radians(90 - bearing)   # convert bearing → math angle
    x = math.cos(angle_math)
    y = math.sin(angle_math)

    # Spoke line
    ax.plot([0, x * 0.9], [0, y * 0.9], color="#888888", linewidth=0.8, zorder=1)

    # Label
    offset = 1.25
    ax.text(
        x * offset, y * offset, label,
        ha="center", va="center",
        fontsize=9,
        fontweight="bold" if bearing % 90 == 0 else "normal",
        color="#1a1a2e" if bearing % 90 == 0 else "#444444",
    )

    # Degree annotation on the circle rim
    ax.text(
        x * 1.05, y * 1.05,
        f"{bearing}°",
        ha="center", va="center",
        fontsize=6.5, color="#777777",
    )

# Center dot
ax.plot(0, 0, "o", color="#e63946", markersize=6, zorder=5)

# North arrow
ax.annotate(
    "", xy=(0, 0.88), xytext=(0, 0.1),
    arrowprops=dict(arrowstyle="-|>", color="#e63946", lw=1.5),
    zorder=6,
)

plt.tight_layout()
plt.show()

## 5. Bearings Between Real Places

Bearing becomes concrete when you attach it to actual geography. The city pairs in the data file give you a feel for what common bearing values look like on a real map.

Each pair starts in **Wichita Falls, TX** and heads somewhere else. Before looking at the approximate bearing, try to guess the direction from the map in your head — then check.

| From | To | Approximate Bearing | What It Means |
|------|----|-------------------|---------------|
| Wichita Falls | Oklahoma City | ~19° | Nearly due North, slight right |
| Wichita Falls | Dallas | ~144° | Southeast — past 90°, not quite South |
| Wichita Falls | Lubbock | ~264° | Nearly due West, just past 270° would be NW |
| Wichita Falls | Altus AFB | ~330° | Northwest — 30° left of due North |

Notice that a bearing of `144°` does not *feel* like "Southeast" at first glance — but 135° is the intercardinal SE, and 144° is just past it. Bearings require practice before they feel intuitive.

In [ ]:
# Print each city pair with its approximate bearing and a rough verbal description
print(f"{'From':<16} {'To':<18} {'Bearing':>8}  Description")
print("-" * 62)
for pair in examples["city_pairs"]:
    frm  = pair["from"]["name"]
    to   = pair["to"]["name"]
    brg  = pair["approximate_bearing"]
    desc = pair["description"]
    print(f"{frm:<16} {to:<18} {brg:>7}°  {desc}")

---

## Exercise A — Name the Bearing

For each bearing below, write the closest compass label (North, NE, East, SE, South, SW, West, NW) without computing anything. Just reason from the circle.

```
  0°  →  ?
 90°  →  ?
180°  →  ?
270°  →  ?
 45°  →  ?
135°  →  ?
315°  →  ?
 22°  →  ?   (hint: between North and NE)
200°  →  ?   (hint: past South, heading toward SW)
```

In [ ]:
answers = {
      0: 'North',
     90: 'East',
    180: 'South',
    270: 'West',
     45: 'NE',
    135: 'SE',
    315: 'NW',
     22: 'North / NNE-ish',
    200: 'South / SSW-ish',
}

for bearing, label in answers.items():
    print(f"{bearing:>4}°  →  {label}")


## Exercise B — Normalize Invalid Values

Bearing must always be in `[0, 360)`. Each value below is the output of a raw `atan2` calculation that forgot to normalize. Write code to fix each one using `(value + 360) % 360`.

```python
raw_bearings = [-45, -90, -180, 380, 400, -1, 720]
```

Print each raw value alongside its normalized result.

In [ ]:
raw_bearings = [-45, -90, -180, 380, 400, -1, 720]

for value in raw_bearings:
    normalized = value % 360
    print(f'{value:>5}° -> {normalized:>5}°')


## Exercise C — Describe the City Pair Bearings

Using the four Wichita Falls city pairs from the data file, answer in plain English — no math required:

1. Wichita Falls → Oklahoma City is bearing `~19°`. Is that closer to North or NE? How would you describe it in one phrase?
2. Wichita Falls → Dallas is bearing `~144°`. What quadrant is that in? What is the nearest intercardinal direction?
3. Wichita Falls → Lubbock is bearing `~264°`. Is that exactly West? What does the extra 6° past 270° tell you?  
   *(Wait — is 264° before or after 270°? Think carefully.)*
4. Wichita Falls → Altus AFB is bearing `~330°`. How far is that from due North (0° or 360°)?

In [ ]:
print('1. OKC: almost due north, just a little east of north.')
print('2. Dallas: in the southeast quadrant, closest to SE.')
print('3. Lubbock: not exactly west; 264° is 6° before 270°, so it is slightly south of due west.')
print('4. Altus AFB: about 30° west of due north, so it is a northwest heading.')


`360` shows up because some code normalizes into a closed interval or rounds a value that is mathematically equivalent to `0°` but lands on the endpoint instead. The minimal fix is to normalize bearings with `% 360` before testing them, so both `360` and `0` collapse to `0` and the north check uses the normalized value.


## Next

In [01 — Compute Bearing](./01-Compute_Bearing.ipynb), we derive the formula — using `math.atan2` to compute the angle between two points and converting from the math convention to the compass convention.